# Week 4 — Retrieval-Augmented Generation (RAG) (Local with LiteLLM & Ollama)

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-04-rag-content.html`. This is the last notebook before your Corte 1 project delivery
in Week 5 — the pipeline you build here is a working draft of that project.

**You will practice:**
1. Chunking a document with overlap.
2. Indexing chunks in ChromaDB, and the same vectors in FAISS for comparison.
3. A complete `answer_with_rag` function using LiteLLM.
4. A light ADK agent with a `retrieve_context` tool, and the LangChain equivalent.
5. Two open exercises — including running this on your **own** project documents.

**Environment:** Running 100% locally with Ollama, RTX GPU acceleration, LiteLLM (`qwen2.5:14b` & `nomic-embed-text`).

In [1]:
%pip install -q --upgrade litellm google-adk langchain-community langchain-ollama python-dotenv numpy chromadb faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [1]:
import litellm
import numpy as np
from dotenv import load_dotenv
from litellm import completion

load_dotenv()
MODEL = "ollama_chat/qwen2.5:14b"
EMBED_MODEL = "ollama/nomic-embed-text"

def embed(text):
    result = litellm.embedding(model=EMBED_MODEL, input=text)
    return result.data[0]["embedding"]

## 0. Sample corpus

A small self-contained knowledge base about this course, so the notebook runs without any external files. In the
lab, swap this for **your own project documents** (see Exercise 1).

In [2]:
COURSE_DOCS = """
AI Agentic Engineering is a 16-week elective course for Systems Engineering students at Universidad de
Santander. It is organized into three graded cuts called cortes. Corte 1 (weeks 1-5) covers LLM fundamentals,
context engineering, and RAG. Corte 2 (weeks 6-11) covers agents, multi-agent systems, Google ADK, and
LangGraph. Corte 3 (weeks 12-16) covers evaluation, observability, and deployment to production.

Corte 1 is worth 30% of the final grade: 25% for a practical project and 5% for in-class activities such as
quizzes, workshops, and labs. The Corte 1 project requires building a conversational assistant that combines
context engineering and RAG over a set of documents chosen by the student, using the Gemini API and ChromaDB,
delivered as a GitHub repository with a live 10-minute demonstration.

Corte 2 is also worth 30%: 25% for a multi-agent system project and 5% for in-class activities. Students must
build a multi-agent system that solves a real problem using Google ADK or LangGraph, integrating RAG
capabilities, delivered with a system diagram and a 15-minute live demonstration.

Corte 3 is worth 40% of the grade: 30% for a final integrator project and 10% for in-class activities. The
final project must combine agents, RAG, an automatic evaluation pipeline, tracing with Langfuse, and a REST
API exposed with FastAPI, delivered with a 5-minute demo video and a 20-minute technical presentation.

All labs in this course use free-tier tools: the Gemini API through Google AI Studio, ChromaDB and FAISS for
vector storage, and Langfuse's free tier for observability starting in Corte 3. Students are also introduced,
at a basic level, to Google Antigravity CLI, a terminal-based coding agent that can scaffold and edit project
files.

Class time each week is split into two blocks: two hours of theory with instructor-led demonstrations and
class discussion, followed by four hours of hands-on lab. Labs mix guided coding, small-group workshops,
individual work, and peer code review. All lab work is submitted through the course GitHub repository, with
Moodle used for supporting material and announcements.
"""

## 1. Chunking

In [3]:
def chunk_text(text: str, chunk_size: int = 400, overlap: int = 60) -> list[str]:
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = chunk_text(COURSE_DOCS, chunk_size=400, overlap=60)
print(f"{len(COURSE_DOCS)} chars -> {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"[{i}] {c[:70]}...")

2137 chars -> 7 chunks
[0] AI Agentic Engineering is a 16-week elective course for Systems Engine...
[1] s 12-16) covers evaluation, observability, and deployment to productio...
[2] s chosen by the student, using the Gemini API and ChromaDB, delivered ...
[3]  capabilities, delivered with a system diagram and a 15-minute live de...
[4]  5-minute demo video and a 20-minute technical presentation. All labs ...
[5] d coding agent that can scaffold and edit project files. Class time ea...
[6]  the course GitHub repository, with Moodle used for supporting materia...


## 2. Indexing with ChromaDB

In [4]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db_week4")
# start fresh each run of this notebook
try:
    chroma_client.delete_collection("course_docs")
except Exception:
    pass
collection = chroma_client.create_collection(name="course_docs")

collection.add(
    documents=chunks,
    embeddings=[embed(c) for c in chunks],
    ids=[f"chunk-{i}" for i in range(len(chunks))],
)
print(collection.count(), "chunks indexed")

7 chunks indexed


### 2a. The same vectors in FAISS (for comparison)

In [6]:
import faiss

vectors = np.array([embed(c) for c in chunks], dtype="float32")
faiss_index = faiss.IndexFlatL2(vectors.shape[1])
faiss_index.add(vectors)

query_vector = np.array([embed("How is Corte 1 graded?")], dtype="float32")
distances, indices = faiss_index.search(query_vector, k=3)
for dist, idx in zip(distances[0], indices[0]):
    print(f"{dist:.3f}  {chunks[idx][:80]}...")

0.566  s 12-16) covers evaluation, observability, and deployment to production. Corte 1...
0.796   capabilities, delivered with a system diagram and a 15-minute live demonstratio...
0.815  s chosen by the student, using the Gemini API and ChromaDB, delivered as a GitHu...


## 3. A complete `answer_with_rag` function

In [7]:
def answer_with_rag(question: str, n_results: int = 3, verbose: bool = False) -> str:
    results = collection.query(query_embeddings=[embed(question)], n_results=n_results)
    retrieved_chunks = results["documents"][0]
    if verbose:
        print("--- retrieved chunks ---")
        for c in retrieved_chunks:
            print("-", c[:80], "...")
    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f'''Answer the question using ONLY the context below. If the answer isn't
in the context, say you don't have that information — do not make anything up.

Context:
{context}  

Question: {question}
Answer:'''

    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
    )
    return response.choices[0].message.content


print(answer_with_rag("What programming language does the course use?"))

The given context does not specify the programming language used in the course.


## 4. Light preview: RAG as an ADK tool, and the LangChain equivalent

We're one step away from a real agent (Week 6 onward covers tool use and ReAct properly). For now, notice that
"retrieval" is just a Python function — which is exactly what an ADK **tool** is.

In [10]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


def retrieve_context(query: str) -> dict:
    """Retrieve the most relevant course-document chunks for a query."""
    results = collection.query(query_embeddings=[embed(query)], n_results=3)
    return {"chunks": results["documents"][0]}

rag_agent = Agent(
    model=LiteLlm(model=MODEL),
    name="course_rag_agent",
    instruction=(
        "Use the retrieve_context tool to find relevant chunks before answering. "
        "Answer ONLY using information returned by the tool. If it's not there, say so."
    ),
    tools=[retrieve_context],
)

async def ask_adk_agent(agent, prompt, app_name="week4_app", user_id="student"):
    session_service = InMemorySessionService()
    runner = Runner(
        agent=agent,
        app_name=app_name,
        session_service=session_service,
        auto_create_session=True,
    )
    session = await session_service.create_session(app_name=app_name, user_id=user_id)
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = ""
    async for event in runner.run_async(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text:
                    final_text += p.text
    return final_text

adk_rag_response = await ask_adk_agent(rag_agent, "How is Corte 2 graded?")
print(adk_rag_response)

Corte 2 is graded based on a project that makes up 25% of the final grade and in-class activities accounting for 5%. The project involves building a multi-agent system that addresses a real-world problem using either Google ADK or LangGraph, with RAG capabilities integrated. The project is delivered with a system diagram and includes a 15-minute live demonstration.


In [9]:
from langchain_ollama import ChatOllama

ollama_model_name = MODEL.replace("ollama_chat/", "").replace("ollama/", "")
llm = ChatOllama(model=ollama_model_name)

def answer_with_rag_langchain(question: str, n_results: int = 3) -> str:
    results = collection.query(query_embeddings=[embed(question)], n_results=n_results)
    context = "\n\n---\n\n".join(results["documents"][0])
    prompt = (
        "Answer the question using ONLY the context below. If the answer isn't in the "
        "context, say you don't have that information.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return llm.invoke(prompt).content

print(answer_with_rag_langchain("What tools are used for observability in this course?"))

The tools used for observability in this course include Langfuse's free tier, which is introduced starting in Corte 3.


## 5. Exercises

In [11]:
# Exercise 1 — Run this on YOUR OWN Corte 1 project documents
# Project: Cubik-Lite, an integral-calculus tutor assistant. Source docs live in
# ../../Project/data (the actual Corte 1 project repo), grouped by topic: indefinite_integrals,
# techniques, definite_integrals, applications. Each doc has LaTeX formulas, so we use a bigger
# chunk_size/overlap than the course sample to reduce the odds of splitting a formula in half.

from pathlib import Path

project_data_dir = Path("../../Project/data")
project_files = sorted(file for file in project_data_dir.rglob("*.md") if file.name != "README.md")

print(f"Found {len(project_files)} project documents:")
for file in project_files:
    print(" -", file.relative_to(project_data_dir))

project_chunks = []
project_sources = []
for file in project_files:
    text = file.read_text(encoding="utf-8")
    file_chunks = chunk_text(text, chunk_size=600, overlap=100)
    project_chunks.extend(file_chunks)
    project_sources.extend([str(file.relative_to(project_data_dir))] * len(file_chunks))

print(f"\n{len(project_chunks)} chunks total from {len(project_files)} files")

proj_collection = chroma_client.create_collection(name="corte1_project_docs")
proj_collection.add(
    documents=project_chunks,
    embeddings=[embed(c) for c in project_chunks],
    metadatas=[{"source": s} for s in project_sources],
    ids=[f"proj-chunk-{i}" for i in range(len(project_chunks))],
)
print(proj_collection.count(), "chunks indexed")

def answer_project_rag(question: str, n_results: int = 3, verbose: bool = False) -> str:
    results = proj_collection.query(query_embeddings=[embed(question)], n_results=n_results)
    retrieved_chunks = results["documents"][0]
    if verbose:
        print("--- retrieved chunks ---")
        for meta, c in zip(results["metadatas"][0], retrieved_chunks):
            print(f"[{meta['source']}]", c[:80], "...")
    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f'''Responde la pregunta usando SOLO el contexto de abajo. Si la respuesta no
    está en el contexto, di que no tienes esa información — no inventes nada.

    Contexto:
    {context}

    Pregunta: {question}
    Respuesta:'''

    res = completion(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.1)
    return res.choices[0].message.content

# 5 real questions a future user of Cubik-Lite (a calculus student) would actually ask
project_questions = [
    "¿Cómo se calcula la integral de x*e^x usando integración por partes?",
    "¿Cuál es la fórmula del Teorema Fundamental del Cálculo?",
    "¿Qué es la regla LIATE y para qué sirve?",
    "¿Cómo se calcula el volumen de un sólido de revolución con el método de discos?",
    "¿Cuándo conviene usar sustitución trigonométrica en vez de sustitución simple?",
]

for q in project_questions:
    print(f"Q: {q}")
    print(f"A: {answer_project_rag(q, verbose=True).strip()}\n")

Found 11 project documents:
 - applications\area_between_curves.md
 - applications\area_under_curve.md
 - applications\volume_of_revolution.md
 - definite_integrals\fundamental_theorem.md
 - definite_integrals\properties.md
 - indefinite_integrals\antiderivatives.md
 - indefinite_integrals\basic_rules.md
 - techniques\integration_by_parts.md
 - techniques\partial_fractions.md
 - techniques\substitution.md
 - techniques\trig_substitution.md

54 chunks total from 11 files
54 chunks indexed
Q: ¿Cómo se calcula la integral de x*e^x usando integración por partes?
--- retrieved chunks ---
[techniques\integration_by_parts.md]  - **Exponenciales:** $e^x,\, 2^x$ > **Regla práctica:** La función que aparezca ...
[indefinite_integrals\basic_rules.md] # Reglas Básicas de Integración La integración indefinida es la operación invers ...
[techniques\integration_by_parts.md] # Integración por Partes El método de **integración por partes** es la contrapar ...
A: Para calcular la integral de \( x e^x \)

In [12]:
# TODO Exercise 2 — Metadata filtering
# Re-index the chunks, but this time attach a `metadatas=[{"source": "..."}, ...]` list to
# collection.add() so each chunk remembers which source document it came from.
# Then query with a `where={"source": "..."}` filter to restrict retrieval to just one document.
# (This is a preview of "RAG avanzado: filtrado por metadata", covered in depth in Week 10.)

docs_by_source = {
    "maintenance_manual.txt": "For hydraulic disc brakes, bleed the lines if the lever feels spongy. Replace pads when under 1.5mm thickness.",
    "pricing_policy.txt": "Electric e-bike rental is $35 per day. Standard commuter is $15 per day. Deposit is $50 refundable.",
}

meta_collection = chroma_client.create_collection(name="filtered_docs")

meta_chunks = []
metadatas = []
ids = []

for src, text in docs_by_source.items():
    src_chunks = chunk_text(text, chunk_size=200, overlap=20)
    for idx, chunk in enumerate(src_chunks):
        meta_chunks.append(chunk)
        metadatas.append({"source": src})
        ids.append(f"{src}-chunk-{idx}")

meta_collection.add(
    documents=meta_chunks,
    embeddings=[embed(c) for c in meta_chunks],
    metadatas=metadatas,
    ids=ids,
)

# Query filtered strictly to pricing_policy.txt
filtered_query = "What is the cost?"
pricing_results = meta_collection.query(
    query_embeddings=[embed(filtered_query)],
    n_results=2,
    where={"source": "pricing_policy.txt"},
)

print("--- Filtered Query Results (source = pricing_policy.txt) ---")
for doc, meta in zip(pricing_results["documents"][0], pricing_results["metadatas"][0]):
    print(f"Source: {meta['source']} | Content: {doc}")

--- Filtered Query Results (source = pricing_policy.txt) ---
Source: pricing_policy.txt | Content: Electric e-bike rental is $35 per day. Standard commuter is $15 per day. Deposit is $50 refundable.


## Corte 1 wrap-up

You now have every piece needed for your Week 5 project: a system prompt (Week 2), context/history handling
(Week 3), and a working RAG pipeline (this week). See `week-04-rag-activities.html` for the full delivery
checklist and grading rubric.

## Looking ahead

Corte 2 starts in Week 6 with agent fundamentals — tool use, function calling, and the ReAct pattern — building
directly on the `tools=[retrieve_context]` preview above.